# 02 — Build the Gleason Child–Caregiver Dataset

This notebook converts the uploaded **Gleason CHILDES corpus** into a canonical dataset for later
single-turn, multi-turn, and LoRA/QLoRA experiments.

The corpus was inspected before this notebook was generated. It contains separate **Mother**,
**Father**, and **Dinner** transcript directories. The CHAT metadata uses `CHI` for the target child,
`MOT` for mother, `FAT` for father, and may also contain investigators or other participants.

## Design decisions

- Keep target children aged **4;00 through 5;11** (`48 <= age_months < 72`).
- Keep only **CHI ↔ MOT/FAT** speech for the core caregiver dataset.
- Exclude investigator, sibling, unidentified, and other participant turns from model training.
- Preserve the original chronological order.
- Retain available CHAT utterance timestamps.
- Represent a missing interlocutor response between consecutive same-role turns as `<SILENCE>`.
- Add a duration only when the neighboring source turns provide usable timestamps.
- Split data **by child**, never randomly by utterance, to reduce identity leakage.
- Export canonical turns, interaction pairs, conversations, and split manifests.


In [1]:
from pathlib import Path
from zipfile import ZipFile
import tempfile
import re
import json
import random

import pandas as pd

ZIP_PATH = Path("Gleason.zip")  # change if needed
OUTPUT_DIR = Path("output_gleason")
OUTPUT_DIR.mkdir(exist_ok=True)

MIN_AGE_MONTHS = 48
MAX_AGE_MONTHS = 72
RANDOM_STATE = 42

# For the core dataset, these are the only caregiver roles.
CAREGIVER_CODES = {"MOT": "mother", "FAT": "father"}


## 1. Extract the corpus and discover CHAT transcripts


In [2]:
if not ZIP_PATH.exists():
    raise FileNotFoundError(f"Cannot find {ZIP_PATH.resolve()}")

extract_dir = Path(tempfile.mkdtemp(prefix="gleason_"))

with ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(extract_dir)

cha_files = sorted(extract_dir.rglob("*.cha"))
print(f"Found {len(cha_files)} CHAT transcripts.")
print("Top-level conditions:", sorted({p.relative_to(extract_dir).parts[0] for p in cha_files}))


Found 71 CHAT transcripts.
Top-level conditions: ['Dinner', 'Father', 'Mother']


## 2. CHAT parsing helpers

This parser intentionally reads the source CHAT text directly. That gives us control over the
participant metadata and the media-time markers embedded on main speaker tiers.

CHAT timing markers have the form `\x15START_END\x15`, where the numeric boundaries are retained
as source timing values. The notebook converts them to seconds assuming the standard millisecond
media-time representation used in these aligned files.


In [3]:
TIME_RE = re.compile(r"\x15(\d+)_(\d+)\x15")
SPEAKER_RE = re.compile(r"^\*([^:]+):\s*(.*)$")
ID_RE = re.compile(r"^@ID:\s*(.*)$")


def parse_age_months(age_text):
    if not age_text:
        return None
    m = re.match(r"^(\d+);(\d+)(?:\.(\d*))?$", age_text.strip())
    if not m:
        return None
    years = int(m.group(1))
    months = int(m.group(2))
    days = int(m.group(3)) if m.group(3) else 0
    return years * 12 + months + days / 30.44


def clean_main_tier(text):
    # Remove media bullets while preserving the linguistic CHAT transcription.
    text = TIME_RE.sub("", text)
    return " ".join(text.split()).strip()


def parse_chat_file(path):
    raw = path.read_text(encoding="utf-8", errors="replace")
    lines = raw.splitlines()

    participants = {}
    situation = None
    media = None

    for line in lines:
        if line.startswith("@Situation:"):
            situation = line.split(":", 1)[1].strip()
        elif line.startswith("@Media:"):
            media = line.split(":", 1)[1].strip()
        else:
            m = ID_RE.match(line)
            if m:
                fields = m.group(1).split("|")
                if len(fields) >= 8:
                    participants[fields[2]] = {
                        "language": fields[0],
                        "corpus": fields[1],
                        "code": fields[2],
                        "age": fields[3],
                        "sex": fields[4],
                        "group": fields[5],
                        "role": fields[7],
                    }

    child = participants.get("CHI", {})
    age_text = child.get("age", "")
    age_months = parse_age_months(age_text)

    turns = []
    for line in lines:
        m = SPEAKER_RE.match(line)
        if not m:
            continue

        speaker = m.group(1)
        content = m.group(2)
        tm = TIME_RE.search(content)

        start_ms = int(tm.group(1)) if tm else None
        end_ms = int(tm.group(2)) if tm else None

        turns.append({
            "speaker_code": speaker,
            "speaker_role_raw": participants.get(speaker, {}).get("role"),
            "text": clean_main_tier(content),
            "start_ms": start_ms,
            "end_ms": end_ms,
        })

    return {
        "participants": participants,
        "child_age_text": age_text,
        "child_age_months": age_months,
        "situation": situation,
        "media": media,
        "turns": turns,
    }


## 3. Audit the corpus before filtering


In [4]:
audit = []

for path in cha_files:
    doc = parse_chat_file(path)
    rel = path.relative_to(extract_dir)

    audit.append({
        "file": str(rel),
        "condition": rel.parts[0],
        "child_id": path.stem,
        "age": doc["child_age_text"],
        "age_months": doc["child_age_months"],
        "speakers": ", ".join(sorted(doc["participants"].keys())),
        "roles": ", ".join(
            f"{k}:{v.get('role')}" for k, v in sorted(doc["participants"].items())
        ),
        "utterances": len(doc["turns"]),
        "timed_utterances": sum(t["start_ms"] is not None for t in doc["turns"]),
    })

audit_df = pd.DataFrame(audit)
display(audit_df.head())
print("\nConditions:")
display(audit_df.groupby("condition").size().rename("transcripts").to_frame())


,file,condition,child_id,age,age_months,speakers,roles,utterances,timed_utterances
0,Dinner\andy.cha,Dinner,andy,4;02.,50.000000,"CHI, FAT, JAN, MOT","CHI:Target_Child, FAT:Father, JAN:Unidentified...",441,441
1,Dinner\bobby.cha,Dinner,bobby,4;02.01,50.032852,"CHI, FAT, MOT, SIS","CHI:Target_Child, FAT:Father, MOT:Mother, SIS:...",1254,1244
2,Dinner\charlie.cha,Dinner,charlie,3;00.01,36.032852,"CHI, FAT, MOT","CHI:Target_Child, FAT:Father, MOT:Mother",960,955
3,Dinner\david.cha,Dinner,david,4;02.01,50.032852,"CHI, FAT, MOT, SIS","CHI:Target_Child, FAT:Father, MOT:Mother, SIS:...",701,700
4,Dinner\eddie.cha,Dinner,eddie,4;04.03,52.098555,"CHI, FAT, MOT","CHI:Target_Child, FAT:Father, MOT:Mother",667,666



Conditions:


,transcripts
condition,
Dinner,23
Father,24
Mother,24


## 4. Select ages 4–6 and identify valid caregiver turns


In [5]:
eligible = audit_df[
    audit_df["age_months"].between(
        MIN_AGE_MONTHS,
        MAX_AGE_MONTHS,
        inclusive="left"
    )
].copy()

print(f"Eligible transcripts: {len(eligible)}")
print(f"Unique eligible children: {eligible['child_id'].nunique()}")
display(
    eligible.groupby(["condition"])
            .agg(transcripts=("file", "count"),
                 children=("child_id", "nunique"))
)


Eligible transcripts: 29
Unique eligible children: 10


,transcripts,children
condition,,
Dinner,10,10
Father,10,10
Mother,9,9


## 5. Build canonical child/caregiver turns

The core dataset contains only `CHI`, `MOT`, and `FAT`. An investigator's turn is **not silently
removed and then treated as if the child and caregiver had spoken directly to each other**.
Instead, excluded speakers break the current interaction segment.

This prevents an exchange such as `CHI → INV → MOT` from becoming a false `CHI → MOT` pair.


In [6]:
turn_rows = []
conversation_rows = []

eligible_files = set(eligible["file"])

for path in cha_files:
    rel = str(path.relative_to(extract_dir))
    if rel not in eligible_files:
        continue

    doc = parse_chat_file(path)
    condition = path.relative_to(extract_dir).parts[0]
    child_id = path.stem

    allowed = {"CHI"} | set(CAREGIVER_CODES)
    segment_id = 0
    kept_turn_index = 0

    for source_index, t in enumerate(doc["turns"]):
        if t["speaker_code"] not in allowed:
            segment_id += 1
            continue

        role = "child" if t["speaker_code"] == "CHI" else "caregiver"
        caregiver_role = CAREGIVER_CODES.get(t["speaker_code"])

        turn_rows.append({
            "corpus": "Gleason",
            "condition": condition,
            "conversation_id": f"Gleason/{condition}/{child_id}",
            "segment_id": segment_id,
            "child_id": child_id,
            "age_text": doc["child_age_text"],
            "age_months": doc["child_age_months"],
            "source_turn_index": source_index,
            "kept_turn_index": kept_turn_index,
            "speaker_code": t["speaker_code"],
            "role": role,
            "caregiver_role": caregiver_role,
            "text": t["text"],
            "start_ms": t["start_ms"],
            "end_ms": t["end_ms"],
            "start_seconds": t["start_ms"] / 1000 if t["start_ms"] is not None else None,
            "end_seconds": t["end_ms"] / 1000 if t["end_ms"] is not None else None,
            "situation": doc["situation"],
            "media": doc["media"],
        })
        kept_turn_index += 1

turns_df = pd.DataFrame(turn_rows)

print(f"Canonical CHI/MOT/FAT turns: {len(turns_df):,}")
display(turns_df.head(20))


Canonical CHI/MOT/FAT turns: 22,712


,corpus,condition,conversation_id,segment_id,child_id,age_text,age_months,source_turn_index,kept_turn_index,speaker_code,role,caregiver_role,text,start_ms,end_ms,start_seconds,end_seconds,situation,media
0,Gleason,Dinner,Gleason/Dinner/andy,0,andy,4;02.,50.0,0,0,CHI,child,NaN,I'll show you which one's yours .,15000.0,17440.0,15.000,17.440,Home Dinner Conversation,"andy, audio"
1,Gleason,Dinner,Gleason/Dinner/andy,0,andy,4;02.,50.0,1,1,CHI,child,NaN,this one's yours .,17440.0,19791.0,17.440,19.791,Home Dinner Conversation,"andy, audio"
2,Gleason,Dinner,Gleason/Dinner/andy,0,andy,4;02.,50.0,2,2,MOT,caregiver,mother,"okay , you wanna sit there xxx ?",19791.0,20805.0,19.791,20.805,Home Dinner Conversation,"andy, audio"
3,Gleason,Dinner,Gleason/Dinner/andy,0,andy,4;02.,50.0,3,3,CHI,child,NaN,"no , I wanna sit in my own .",20805.0,22374.0,20.805,22.374,Home Dinner Conversation,"andy, audio"
4,Gleason,Dinner,Gleason/Dinner/andy,0,andy,4;02.,50.0,4,4,CHI,child,NaN,"zum@b , zum@b .",22374.0,23993.0,22.374,23.993,Home Dinner Conversation,"andy, audio"
5,Gleason,Dinner,Gleason/Dinner/andy,0,andy,4;02.,50.0,5,5,CHI,child,NaN,xxx from my own place !,23993.0,26152.0,23.993,26.152,Home Dinner Conversation,"andy, audio"
6,Gleason,Dinner,Gleason/Dinner/andy,0,andy,4;02.,50.0,6,6,CHI,child,NaN,I xxx in my own place !,26152.0,28300.0,26.152,28.300,Home Dinner Conversation,"andy, audio"
7,Gleason,Dinner,Gleason/Dinner/andy,0,andy,4;02.,50.0,7,7,CHI,child,NaN,place !,28300.0,29498.0,28.300,29.498,Home Dinner Conversation,"andy, audio"
8,Gleason,Dinner,Gleason/Dinner/andy,0,andy,4;02.,50.0,8,8,CHI,child,NaN,xxx .,29498.0,31522.0,29.498,31.522,Home Dinner Conversation,"andy, audio"
9,Gleason,Dinner,Gleason/Dinner/andy,0,andy,4;02.,50.0,9,9,MOT,caregiver,mother,xxx okay ?,31522.0,33296.0,31.522,33.296,Home Dinner Conversation,"andy, audio"


## 6. Build alternating interaction pairs with `<SILENCE>`

Within each uninterrupted child/caregiver segment:

- `caregiver → child` becomes a caregiver-prompt/child-response pair.
- `child → caregiver` becomes a child-prompt/caregiver-response pair.
- If the same role speaks twice consecutively, the absent role receives `<SILENCE>`.
- When timestamps exist for both neighboring turns, `silence_seconds` records the observed gap.
- We retain `<SILENCE>` as the model token rather than embedding continuous durations into the
  vocabulary. Duration remains a numeric feature that can be discretized later if your experiment
  supports it.

This is safer for the first prototype than creating thousands of distinct
`<SILENCE FOR X SECONDS>` strings.


In [7]:
pair_rows = []

group_cols = ["conversation_id", "segment_id"]

for (conversation_id, segment_id), group in turns_df.groupby(group_cols, sort=False):
    group = group.sort_values("source_turn_index").reset_index(drop=True)

    for i in range(len(group) - 1):
        a = group.iloc[i]
        b = group.iloc[i + 1]

        # Source gap, if both utterances have aligned media timing.
        gap_seconds = None
        if pd.notna(a["end_ms"]) and pd.notna(b["start_ms"]):
            gap_seconds = max(0.0, (b["start_ms"] - a["end_ms"]) / 1000)

        common = {
            "corpus": a["corpus"],
            "condition": a["condition"],
            "conversation_id": conversation_id,
            "segment_id": segment_id,
            "child_id": a["child_id"],
            "age_months": a["age_months"],
            "source_gap_seconds": gap_seconds,
        }

        if a["role"] != b["role"]:
            pair_rows.append({
                **common,
                "prompt_role": a["role"],
                "prompt": a["text"],
                "response_role": b["role"],
                "response": b["text"],
                "response_is_silence": False,
            })
        else:
            missing_role = "caregiver" if a["role"] == "child" else "child"
            pair_rows.append({
                **common,
                "prompt_role": a["role"],
                "prompt": a["text"],
                "response_role": missing_role,
                "response": "<SILENCE>",
                "response_is_silence": True,
            })

pairs_df = pd.DataFrame(pair_rows)

print(f"Interaction pairs: {len(pairs_df):,}")
print(f"Inserted <SILENCE> responses: {pairs_df['response_is_silence'].sum():,}")
display(pairs_df.head(20))


Interaction pairs: 21,994
Inserted <SILENCE> responses: 10,111


,corpus,condition,conversation_id,segment_id,child_id,age_months,source_gap_seconds,prompt_role,prompt,response_role,response,response_is_silence
0,Gleason,Dinner,Gleason/Dinner/andy,0,andy,50.0,0.0,child,I'll show you which one's yours .,caregiver,<SILENCE>,True
1,Gleason,Dinner,Gleason/Dinner/andy,0,andy,50.0,0.0,child,this one's yours .,caregiver,"okay , you wanna sit there xxx ?",False
2,Gleason,Dinner,Gleason/Dinner/andy,0,andy,50.0,0.0,caregiver,"okay , you wanna sit there xxx ?",child,"no , I wanna sit in my own .",False
3,Gleason,Dinner,Gleason/Dinner/andy,0,andy,50.0,0.0,child,"no , I wanna sit in my own .",caregiver,<SILENCE>,True
4,Gleason,Dinner,Gleason/Dinner/andy,0,andy,50.0,0.0,child,"zum@b , zum@b .",caregiver,<SILENCE>,True
5,Gleason,Dinner,Gleason/Dinner/andy,0,andy,50.0,0.0,child,xxx from my own place !,caregiver,<SILENCE>,True
6,Gleason,Dinner,Gleason/Dinner/andy,0,andy,50.0,0.0,child,I xxx in my own place !,caregiver,<SILENCE>,True
7,Gleason,Dinner,Gleason/Dinner/andy,0,andy,50.0,0.0,child,place !,caregiver,<SILENCE>,True
8,Gleason,Dinner,Gleason/Dinner/andy,0,andy,50.0,0.0,child,xxx .,caregiver,xxx okay ?,False
9,Gleason,Dinner,Gleason/Dinner/andy,0,andy,50.0,0.0,caregiver,xxx okay ?,child,noodles +...,False


## 7. Dataset diagnostics

Before fine-tuning, inspect how much data comes from each age, condition, and response direction.
This is also where you should detect a corpus imbalance before it becomes a model-training bias.


In [8]:
diagnostics = {
    "unique_children": turns_df["child_id"].nunique(),
    "conversations": turns_df["conversation_id"].nunique(),
    "turns": len(turns_df),
    "pairs": len(pairs_df),
    "child_turns": int((turns_df["role"] == "child").sum()),
    "caregiver_turns": int((turns_df["role"] == "caregiver").sum()),
    "silence_pairs": int(pairs_df["response_is_silence"].sum()),
    "timed_turn_fraction": turns_df["start_ms"].notna().mean(),
}
display(pd.DataFrame([diagnostics]).round(3))

display(
    turns_df.groupby(["condition", "role"])
            .size()
            .unstack(fill_value=0)
)

display(
    pairs_df.groupby(["prompt_role", "response_role"])
            .size()
            .rename("pairs")
            .to_frame()
)


,unique_children,conversations,turns,pairs,child_turns,caregiver_turns,silence_pairs,timed_turn_fraction
0,10,29,22712,21994,8253,14459,10111,0.721


role,caregiver,child
condition,,
Dinner,4408,2266
Father,4957,3009
Mother,5094,2978


,,pairs
prompt_role,response_role,
caregiver,child,13959
child,caregiver,8035


## 8. Child-level train/validation/test split

The split unit is **child identity**, not utterance or transcript. Thus, if the same child appears
in Mother, Father, and Dinner sessions, all of those sessions remain in the same split.

With a small corpus such as Gleason, exact 80/10/10 proportions can be unstable. This function
uses approximately 80/10/10 while guaranteeing at least one validation and one test child when
there are enough children.


In [9]:
children = sorted(turns_df["child_id"].unique())
rng = random.Random(RANDOM_STATE)
rng.shuffle(children)

n = len(children)

if n < 3:
    raise ValueError("Need at least 3 unique children for train/validation/test splitting.")

n_test = max(1, round(n * 0.10))
n_val = max(1, round(n * 0.10))
n_train = n - n_val - n_test

train_children = set(children[:n_train])
val_children = set(children[n_train:n_train + n_val])
test_children = set(children[n_train + n_val:])

def assign_split(child_id):
    if child_id in train_children:
        return "train"
    if child_id in val_children:
        return "validation"
    return "test"

turns_df["split"] = turns_df["child_id"].map(assign_split)
pairs_df["split"] = pairs_df["child_id"].map(assign_split)

split_manifest = (
    turns_df[["child_id", "age_months", "split"]]
    .drop_duplicates("child_id")
    .sort_values(["split", "age_months", "child_id"])
)

display(split_manifest)
display(
    turns_df.groupby("split").agg(
        children=("child_id", "nunique"),
        conversations=("conversation_id", "nunique"),
        turns=("text", "size"),
    )
)


,child_id,age_months,split
430,bobby,50.032852,test
5193,wanda,48.657030,train
1529,david,50.032852,train
4161,john,50.755585,train
4753,theresa,50.985545,train
2229,eddie,52.098555,train
3415,helen,52.459921,train
6058,xavia,53.000000,train
2896,frank,62.229961,train
0,andy,50.000000,validation


,children,conversations,turns
split,,,
test,1,3,2768
train,8,23,18265
validation,1,3,1679


## 9. Create SFT examples

For the first child-model experiment, train only examples where the desired response role is
`child`. Caregiver-response examples are retained separately because they are useful for the
caregiver agent in dynamic testing.

The prompt includes exact age in months so the model can learn age-conditioned behavior.


In [10]:
def child_sft_record(row):
    instruction = (
        f"You are simulating a {row['age_months']:.1f}-month-old child "
        "in a conversation with a caregiver. Respond as the child only."
    )
    return {
        "instruction": instruction,
        "input": f"Caregiver: {row['prompt']}",
        "output": row["response"],
        "age_months": row["age_months"],
        "child_id": row["child_id"],
        "conversation_id": row["conversation_id"],
        "condition": row["condition"],
        "split": row["split"],
        "is_silence": bool(row["response_is_silence"]),
    }

child_pairs = pairs_df[
    (pairs_df["prompt_role"] == "caregiver") &
    (pairs_df["response_role"] == "child")
].copy()

child_sft = [child_sft_record(row) for _, row in child_pairs.iterrows()]

print(f"Child-model SFT examples: {len(child_sft):,}")
child_sft[:3]


Child-model SFT examples: 13,959


[{'instruction': 'You are simulating a 50.0-month-old child in a conversation with a caregiver. Respond as the child only.',
  'input': 'Caregiver: okay , you wanna sit there xxx ?',
  'output': 'no , I wanna sit in my own .',
  'age_months': 50.0,
  'child_id': 'andy',
  'conversation_id': 'Gleason/Dinner/andy',
  'condition': 'Dinner',
  'split': 'validation',
  'is_silence': False},
 {'instruction': 'You are simulating a 50.0-month-old child in a conversation with a caregiver. Respond as the child only.',
  'input': 'Caregiver: xxx okay ?',
  'output': 'noodles +...',
  'age_months': 50.0,
  'child_id': 'andy',
  'conversation_id': 'Gleason/Dinner/andy',
  'condition': 'Dinner',
  'split': 'validation',
  'is_silence': False},
 {'instruction': 'You are simulating a 50.0-month-old child in a conversation with a caregiver. Respond as the child only.',
  'input': 'Caregiver: noodles , noodles .',
  'output': 'mm .',
  'age_months': 50.0,
  'child_id': 'andy',
  'conversation_id': 'Glea

## 10. Export canonical data

Outputs:

- `gleason_turns_age4_6.csv` — canonical source turns
- `gleason_interaction_pairs_age4_6.csv` — paired interactions + silence markers
- `gleason_child_split_manifest.csv` — child-level split assignment
- `gleason_child_sft_{train,validation,test}.jsonl` — child-model examples
- `gleason_conversations.jsonl` — ordered conversations for dynamic evaluation


In [11]:
turns_df.to_csv(OUTPUT_DIR / "gleason_turns_age4_6.csv", index=False)
pairs_df.to_csv(OUTPUT_DIR / "gleason_interaction_pairs_age4_6.csv", index=False)
split_manifest.to_csv(OUTPUT_DIR / "gleason_child_split_manifest.csv", index=False)

for split in ["train", "validation", "test"]:
    path = OUTPUT_DIR / f"gleason_child_sft_{split}.jsonl"
    with path.open("w", encoding="utf-8") as f:
        for rec in child_sft:
            if rec["split"] == split:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")

conversation_path = OUTPUT_DIR / "gleason_conversations.jsonl"
with conversation_path.open("w", encoding="utf-8") as f:
    for conversation_id, group in turns_df.groupby("conversation_id", sort=False):
        group = group.sort_values("source_turn_index")
        first = group.iloc[0]
        rec = {
            "corpus": "Gleason",
            "conversation_id": conversation_id,
            "child_id": first["child_id"],
            "age_months": first["age_months"],
            "condition": first["condition"],
            "split": first["split"],
            "turns": [
                {
                    "role": r["role"],
                    "speaker_code": r["speaker_code"],
                    "caregiver_role": r["caregiver_role"] if pd.notna(r["caregiver_role"]) else None,
                    "text": r["text"],
                    "start_ms": int(r["start_ms"]) if pd.notna(r["start_ms"]) else None,
                    "end_ms": int(r["end_ms"]) if pd.notna(r["end_ms"]) else None,
                }
                for _, r in group.iterrows()
            ],
        }
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print("Exported files:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" -", p)


Exported files:
 - output_gleason\gleason_child_sft_test.jsonl
 - output_gleason\gleason_child_sft_train.jsonl
 - output_gleason\gleason_child_sft_validation.jsonl
 - output_gleason\gleason_child_split_manifest.csv
 - output_gleason\gleason_conversations.jsonl
 - output_gleason\gleason_interaction_pairs_age4_6.csv
 - output_gleason\gleason_turns_age4_6.csv


## 11. Final validation checks

Do not begin LoRA training unless all assertions below pass.


In [12]:
assert set(turns_df["speaker_code"]).issubset({"CHI", "MOT", "FAT"})
assert turns_df["age_months"].between(48, 72, inclusive="left").all()
assert not (train_children & val_children)
assert not (train_children & test_children)
assert not (val_children & test_children)
assert set(turns_df["split"]) == {"train", "validation", "test"}

# Every interaction pair should have one of the intended role directions.
valid_directions = {
    ("child", "caregiver"),
    ("caregiver", "child"),
}
assert all(
    (a, b) in valid_directions
    for a, b in zip(pairs_df["prompt_role"], pairs_df["response_role"])
)

print("All validation checks passed.")
print("\nNext experiment:")
print("1. Establish an off-the-shelf base-model baseline on the TEST children.")
print("2. Run single-turn generation.")
print("3. Run dynamic child-agent ↔ caregiver-agent generation.")
print("4. Fine-tune the child model with LoRA using TRAIN only.")
print("5. Tune decisions on VALIDATION only.")
print("6. Re-run the identical TEST benchmark.")


All validation checks passed.

Next experiment:
1. Establish an off-the-shelf base-model baseline on the TEST children.
2. Run single-turn generation.
3. Run dynamic child-agent ↔ caregiver-agent generation.
4. Fine-tune the child model with LoRA using TRAIN only.
5. Tune decisions on VALIDATION only.
6. Re-run the identical TEST benchmark.
